# Vintage CORE Evaluation

Run one base model on full **Original CORE**, **Filtered CORE**, and **Restyled CORE** from the `Vintage-CORE` branch of `zachnorton14/think.nano`. Original CORE has 22 tasks; the Vintage bundles have 20, so the notebook reports both each bundle's native aggregate and a directly comparable Common-20 aggregate. LAMBADA is informational and never triggers fallback or blocks a result.

## 1. Configuration
Run this notebook once per model. GPT-1900 d34 should use an A100-class Colab runtime.

In [ ]:
MODEL_ID = "think-d12-r30"
RUN_FULL = True
LOG_TO_WANDB = False

VALID_MODELS = ["think-d12-r30", "modern-d24", "gpt1900-d34"]
assert MODEL_ID in VALID_MODELS
assert RUN_FULL is True, "The publication runs are full evaluations."
print("Valid model IDs:", ", ".join(VALID_MODELS))

## 2. Environment
Select **Runtime → Change runtime type → GPU** first. While the dataset is private, add an `HF_TOKEN` secret in Colab (key icon at left).

In [ ]:
import os, platform, subprocess, sys
import torch

assert torch.cuda.is_available(), "A CUDA GPU is required."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"VRAM: {props.total_memory / 2**30:.1f} GiB")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
if MODEL_ID == "gpt1900-d34" and "A100" not in props.name:
    print("WARNING: GPT-1900 d34 is intended for an A100-class runtime.")

REPO_URL = "https://github.com/zachnorton14/think.nano.git"
REPO_BRANCH = "Vintage-CORE"
repo = "/content/think.nano"
if not os.path.exists(repo):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, repo], check=True)
branch = subprocess.check_output(["git", "-C", repo, "branch", "--show-current"], text=True).strip()
assert branch == REPO_BRANCH, f"Expected {REPO_BRANCH}, found {branch}"
print(f"Repository: {REPO_URL} @ {branch}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{repo}/dev/vintage_core_colab/requirements.txt"], check=True)

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass
print("HF_TOKEN available:", bool(os.environ.get("HF_TOKEN")))

## 3. Download, validate, load, and evaluate
The evaluator downloads only the selected checkpoint, metadata, tokenizer, and pinned runtime. It validates every bundle file, runs one forward-pass loader check, then evaluates the three full bundles sequentially. A JSON result is saved after each bundle.

In [ ]:
RESULTS_DIR = "/content/vintage-core-results"
command = [
    sys.executable, f"{repo}/dev/vintage_core_colab/vintage_core_eval.py",
    "--model", MODEL_ID,
    "--bundles", "original,filtered,restyled",
    "--output-dir", RESULTS_DIR,
    "--max-per-task", "-1",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

## 4. Results

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(f"{RESULTS_DIR}/summary.csv")
summary["bundle"] = summary["bundle"].str.title()
summary["runtime"] = pd.to_timedelta(summary.pop("runtime_seconds"), unit="s")
summary = summary.rename(columns={
    "model": "Model", "bundle": "Bundle",
    "native_core": "Native CORE", "common_20_core": "Common-20 CORE",
    "runtime": "Runtime",
})
display(summary.style.format({"Native CORE": "{:.4f}", "Common-20 CORE": "{:.4f}"}))

In [ ]:
accuracy = pd.read_csv(f"{RESULTS_DIR}/task_accuracy.csv").set_index("task")
deltas = pd.read_csv(f"{RESULTS_DIR}/task_deltas.csv").set_index("task")

def highlight_lambada(row):
    return ["background-color: #fff3b0; font-weight: bold" if row.name == "lambada_openai" else "" for _ in row]

print("Raw per-task accuracy")
display(accuracy.style.apply(highlight_lambada, axis=1).format("{:.4f}"))
print("Accuracy deltas")
display(deltas.style.apply(highlight_lambada, axis=1).format("{:+.4f}"))

## 5. Optional W&B logging and download
W&B is not required. The local JSON and CSV files are the canonical notebook outputs.

In [ ]:
if LOG_TO_WANDB:
    import wandb
    run = wandb.init(project="vintage-core", name=f"{MODEL_ID}-core-matrix")
    wandb.log({"core_comparison": wandb.Table(dataframe=summary)})
    run.finish()

# Uncomment in Colab to download all small result files.
# from google.colab import files
# archive = __import__("shutil").make_archive("/content/vintage-core-results", "zip", RESULTS_DIR)
# files.download(archive)